# 推理服务与量化补充线 · 第 6/8 课：量化基础：Scale、Zero Point、Granularity 与校准

> 状态：**未开始**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现对称 per-tensor INT8 fake quantization，并解释 clipping、rounding 与校准分布。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`train/lesson14` 已讲浮点范围与混合精度；本课补推理整数/低比特量化的数据变换和误差来源。

前置：train 第 1～5 课、CUDA/Triton 基础、Transformer attention。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

量化把浮点值映射到有限整数格点。对称量化常用 `scale=max_abs/qmax`、`q=clip(round(x/scale))`，反量化为 `q·scale`。

### 数据与控制如何流动

校准先从代表性数据统计范围并确定 scale/zero-point；转换阶段按粒度量化并保存元数据；运行时 kernel 读取整数值和元数据完成整数计算或融合反量化。

### 正确性条件与常见误区

零张量需特殊处理 scale，rounding 规则必须与目标 kernel 一致；per-channel/group 的 axis 和 packing metadata 是模型格式的一部分。校准集必须代表部署分布。

### 性能、成本与工程取舍

更细粒度 scale 降误差但增加 scale 存储、加载和 kernel 复杂度；动态量化适应输入但每批计算 scale，静态量化更快却怕分布漂移。

## 具体演示

x=[-2,-1,0,1,2]、int8 qmax=127，scale=2/127；端点精确，其他值舍入到邻近格点。若有一个 100 的 outlier，绝大多数小值分辨率会恶化。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐对称 fake quantization；返回反量化值与 scale。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def symmetric_fake_quant(values, qmax=127):
    if qmax <= 0:
        raise ValueError("qmax must be positive")
    if not values:
        return [], 1.0
    max_abs = max(abs(x) for x in values)
    scale = max_abs / qmax if max_abs else 1.0
    quantized = [max(-qmax, min(qmax, round(x / scale))) for x in values]
    # TODO：反量化回浮点格点。
    return ______, scale

dq, scale = symmetric_fake_quant([-2., -1., 0., 1., 2.])
assert abs(dq[0] + 2.0) < 1e-12 and abs(dq[-1] - 2.0) < 1e-12
assert symmetric_fake_quant([0., 0.]) == ([0.0, 0.0], 1.0)


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

per-channel 为什么通常比 per-tensor 精确？代价是什么？

**你的答案：**


### Q2

校准集只含短英文 prompt，部署转成长中文/代码请求，会发生什么？

**你的答案：**


### Q3

fake quantization 的输出还是 float，为什么它不能证明实际 INT4 kernel 性能？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考资料

- [torchao quantized inference](https://docs.pytorch.org/ao/stable/workflows/inference.html)
- [SmoothQuant](https://arxiv.org/abs/2211.10438)

API 与平台能力会演进；部署前应按目标版本重新核对。